<a href="https://colab.research.google.com/github/Jayash05/TruthLens/blob/main/TruthLens.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')


In [ ]:
!pip install torch torchvision transformers datasets pandas numpy scikit-learn
!pip install "ollama" # or use API-based models
!pip install requests beautifulsoup4 selenium # web scraping
!pip install newsapi groq # news API & LLM API
!pip install pillow opencv-python scikit-image # image processing
!pip install matplotlib seaborn plotly # visualization


In [ ]:
!pip install feedparser
import feedparser

def fetch_google_news(topic, max_articles=10):
    query = topic.replace(" ", "+")
    rss_url = f"https://news.google.com/rss/search?q={query}&hl=en-IN&gl=IN&ceid=IN:en"

    feed = feedparser.parse(rss_url)
    articles = []

    for entry in feed.entries[:max_articles]:
        articles.append({
            "source": entry.source.title if hasattr(entry, "source") else "google_news",
            "url": entry.link,
            "headline": entry.title,
            "body_text": entry.summary if hasattr(entry, "summary") else "",
            "image_url": None,  # Google RSS often omits images
            "pub_date": entry.published if hasattr(entry, "published") else None
        })

    return articles


In [ ]:
def fetch_news_by_topic(topic, num_days=1):

    print("NewsData.io...")
    articles = []

    # ---------- Attempt 1: NewsData.io ----------
    try:
        url = "https://newsdata.io/api/1/news"
        params = {
            "q": topic,
            "apikey": "API_key",
            "language": "en"
        }
        r = requests.get(url, params=params, timeout=10)
        data = r.json()

        if isinstance(data.get("results"), list):
            for res in data["results"]:
                articles.append({
                    "source": res.get("source_id", "newsdata"),
                    "url": res.get("link", ""),
                    "headline": res.get("title", ""),
                    "body_text": res.get("description", ""),
                    "image_url": res.get("image_url"),
                    "pub_date": res.get("pubDate")
                })
    except Exception:
        pass

    if articles:
        print(f" NewsData.io returned {len(articles)} articles")
        return articles

    # ---------- Attempt 2: Google News ----------
    print(" Falling back to Google News RSS...")
    articles = fetch_google_news(topic)

    if articles:
        print(f" Google News returned {len(articles)} articles")
        return articles

    # ---------- Attempt 3: Keyword expansion ----------
    print(" Trying keyword expansion fallback...")
    expanded_topics = [
        f"{topic} Andhra Pradesh",
        f"{topic} India",
        "Quantum technology Andhra Pradesh",
        "Amaravati technology hub"
    ]

    for t in expanded_topics:
        articles = fetch_google_news(t)
        if articles:
            print(f"Found articles using expanded query: {t}")
            return articles

    print(" No articles found from any source.")
    return []


In [ ]:
from sentence_transformers import SentenceTransformer, util

def cluster_articles_by_topic(articles, similarity_threshold=0.7):
    if len(articles) <= 2:
        print("Few articles found – using fallback single cluster.")
        return [articles]

    embedder = SentenceTransformer('all-MiniLM-L6-v2')
    headlines = [a["headline"] for a in articles if a.get("headline")]
    embeddings = embedder.encode(headlines, convert_to_tensor=True)

    clusters = []
    used = set()

    for i, emb in enumerate(embeddings):
        if i in used:
            continue

        cluster = [articles[i]]
        used.add(i)

        for j in range(i + 1, len(embeddings)):
            if j not in used:
                similarity = util.pytorch_cos_sim(emb, embeddings[j])[0][0].item()
                if similarity >= similarity_threshold:
                    cluster.append(articles[j])
                    used.add(j)

        clusters.append(cluster)

    return clusters


In [ ]:
import os
from PIL import Image
import requests
from io import BytesIO

def download_and_process_images(article_cluster, max_size=(512, 512)):

    os.makedirs("./downloaded_images", exist_ok=True)

    for article in article_cluster:
        if not article.get("image_url"):
            continue

        try:
            response = requests.get(article["image_url"], timeout=5)
            img = Image.open(BytesIO(response.content))

            if img.size[0] < 100 or img.size[1] < 100:
                continue

            img.thumbnail(max_size, Image.Resampling.LANCZOS)

            filename = f"./downloaded_images/{article['source']}_{len(article_cluster)}_{hash(article['url'])}.jpg"
            img.save(filename, quality=85)
            article["local_image_path"] = filename

        except Exception as e:
            print(f"Failed to download image from {article['url']}: {e}")


In [ ]:
import re
from nltk.tokenize import sent_tokenize

def preprocess_text(text):
    """Clean and normalize text for analysis."""
    if text is None:
        return ""

    text = str(text)

    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'http\S+|www.\S+', '', text)
    text = re.sub(r'[.!?]{2,}', r'.', text)

    return text

def split_into_sentences(text):
    """
    Tokenize article into sentences for granular analysis.
    """
    return sent_tokenize(text)


In [ ]:
import spacy
from groq import Groq

try:
    nlp = spacy.load("en_core_web_sm")
except:
    !python -m spacy download en_core_web_sm
    nlp = spacy.load("en_core_web_sm")

def extract_claims(headline, article_body):
    client = Groq(api_key="API_key")

    prompt = f"""
    Extract all verifiable, atomic claims from this news article.
    A claim must be:
    - A single, testable statement (not an opinion)
    - Specific with dates, numbers, names, or locations
    - Decontextualized (include full context)

    Headline: {headline}
    Article: {article_body[:2000]}

    Format:
    CLAIM 1: [statement]
    CLAIM 2: [statement]
    """

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=500
    )

    raw_claims = []
    for line in response.choices[0].message.content.split('\n'):
        if line.startswith('CLAIM'):
            claim_text = line.split(':', 1)[1].strip()
            # Only keep if NER finds trust signals (Names, Dates, Places)
            doc = nlp(claim_text)
            trust_labels = {"PERSON", "ORG", "GPE", "DATE", "CARDINAL", "MONEY"}
            if any(ent.label_ in trust_labels for ent in doc.ents):
                raw_claims.append(claim_text)

    return raw_claims

In [ ]:
from transformers import pipeline

def analyze_emotion_per_sentence(sentences):
    classifier = pipeline("zero-shot-classification",
                         model="facebook/bart-large-mnli")

    emotions = ["joy", "anger", "fear", "sadness", "neutral"]

    emotion_scores = []
    for sent in sentences:
        if len(sent) < 10:
            continue
        result = classifier(sent, emotions)
        emotion_scores.append({
            "sentence": sent,
            "dominant_emotion": result['labels'][0],
            "scores": dict(zip(result['labels'], result['scores']))
        })

    return emotion_scores

def compute_article_emotional_intensity(emotion_scores):
    if not emotion_scores:
        return None

    # Compute distribution across emotions
    emotion_counts = {e: 0 for e in ["joy", "anger", "fear", "sadness", "neutral"]}
    for es in emotion_scores:
        emotion_counts[es["dominant_emotion"]] += 1

    total = sum(emotion_counts.values())
    emotion_distribution = {e: emotion_counts[e] / total for e in emotion_counts}

    # Sensationalism index: (anger + fear + joy) / total
    sensationalism_score = (emotion_counts["anger"] + emotion_counts["fear"] + emotion_counts["joy"]) / total

    # Neutrality score: inverse of sensationalism
    neutrality_score = 1 - sensationalism_score

    return {
        "emotion_distribution": emotion_distribution,
        "sensationalism_score": sensationalism_score,
        "neutrality_score": neutrality_score,
        "avg_emotion_intensity": sum([max(es["scores"].values()) for es in emotion_scores]) / len(emotion_scores)
    }


In [ ]:
def detect_bias_markers(article_text):
    client = Groq(api_key="api_key")

    prompt = f"""
    Identify bias and sensationalism markers in this news text.
    Score on a scale 1-10 where 1=neutral/factual, 10=highly sensational/biased.

    Look for:
    - Loaded adjectives (e.g., "shocking", "outrageous")
    - Extreme language (e.g., "devastating", "catastrophic")
    - Rhetorical questions meant to sway opinion
    - Unnamed sources ("some say", "experts claim")
    - Emotional appeals instead of facts

    Text: {article_text[:1500]}

    Response format:
    BIAS_SCORE: [1-10]
    MARKERS: [list each detected marker]
    EXPLANATION: [why you gave this score]
    """

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=300
    )

    # Parse response
    output = response.choices[0].message.content
    bias_score = int(re.search(r'BIAS_SCORE:\s*(\d+)', output).group(1))

    return {
        "bias_score": bias_score / 10,  # Normalize to 0-1
        "raw_response": output
    }


In [ ]:
def compute_text_features(article):
    sentences = split_into_sentences(article["body_text"])
    claims = extract_claims(article["headline"], article["body_text"])
    emotions = analyze_emotion_per_sentence(sentences)
    bias = detect_bias_markers(article["body_text"])
    emotional_intensity = compute_article_emotional_intensity(emotions)

    article["text_features"] = {
        "num_claims": len(claims),
        "claims": claims,
        "num_sentences": len(sentences),
        "avg_sentence_length": sum(len(s.split()) for s in sentences) / len(sentences) if sentences else 0,
        "emotion_analysis": emotional_intensity,
        "bias_score": bias["bias_score"],
        "sensationalism_score": emotional_intensity["sensationalism_score"],
        "neutrality_score": emotional_intensity["neutrality_score"]
    }

    return article


In [ ]:
!pip install -U transformers accelerate

from PIL import Image
import torch
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration

device = "cpu"

model_name = "Qwen/Qwen2-VL-2B-Instruct"
processor = AutoProcessor.from_pretrained(model_name, trust_remote_code=True)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.float32,
    trust_remote_code=True
).to(device)

def analyze_image_forensics_with_vlm(image_path, headline):
    """
    Upgraded VLM analysis focusing on context and sensationalism.
    """
    image = Image.open(image_path).convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {
                    "type": "text",
                    "text": (
                        f"Article Headline: {headline}\n\n"
                        "Task: Analyze this news image for veracity.\n"
                        "1. CLASSIFICATION: Is this a stock photo, a real event photo, or AI-generated?\n"
                        "2. ALIGNMENT: Does the visual content directly support the headline, or is it generic/unrelated?\n"
                        "3. SENSATIONALISM: Are there visual manipulations (heavy filters, dramatic angles, exaggerated expressions)?\n"
                        "4. SCORE: Provide an 'Alignment Score' from 1 to 10."
                    )
                }
            ]
        }
    ]

    prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=prompt, images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=300, do_sample=False)

    full_response = processor.batch_decode(output_ids, skip_special_tokens=True)[0]

    try:
        score_match = re.search(r'SCORE:\s*(\d+)', full_response)
        alignment_score = int(score_match.group(1)) / 10.0 if score_match else 0.5
    except:
        alignment_score = 0.5

    return {
        "vlm_analysis": full_response,
        "alignment_score": alignment_score
    }

In [ ]:
import cv2
import numpy as np
from skimage.color import rgb2lab

def analyze_color_emotion(image_path):
    """
    Extract L*a*b* color features and map to emotion dimensions.
    Based on color-emotion research [web:62].

    Key findings:
    - Red + high chroma → joy, rage, dominance
    - High lightness → serenity, relief
    - Low lightness + high arousal → threat
    """
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_lab = rgb2lab(img_rgb / 255.0)

    L_mean = img_lab[:, :, 0].mean()  # Lightness (0-100)
    a_mean = img_lab[:, :, 1].mean()  # Red-green (-128 to 127)
    b_mean = img_lab[:, :, 2].mean()  # Yellow-blue (-128 to 127)

    lightness = L_mean / 100.0
    redness = (a_mean + 128) / 256.0
    yellowness = (b_mean + 128) / 256.0

    chroma = np.sqrt(a_mean**2 + b_mean**2) / 128.0
    emotion_mapping = {
        "valence": lightness + (chroma * 0.3),  # High lightness + saturation = positive
        "arousal": chroma + (1 - lightness) * 0.2,  # High chroma + darkness = arousal
        "dominance": redness + chroma,  # Red + saturation = dominance
    }

    return {
        "color_features": {
            "lightness": lightness,
            "redness": redness,
            "yellowness": yellowness,
            "chroma": min(chroma, 1.0)  # Cap at 1.0
        },
        "emotion_dimensions": emotion_mapping,
        "sensationalism_from_color": chroma * (1 - lightness)  # Dark+saturated = sensational
    }


In [ ]:
from transformers import CLIPProcessor, CLIPModel
import torch
from scipy.spatial.distance import cosine

def compute_image_embeddings_clip(image_paths):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"CLIP using device: {device}")

    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)

    embeddings = []
    for path in image_paths:
        try:
            image = Image.open(path).convert("RGB")
            inputs = processor(images=image, return_tensors="pt").to(device)

            with torch.no_grad():
                outputs = model.get_image_features(**inputs)

                emb = outputs.detach().cpu().numpy().flatten()

            embeddings.append(emb)
        except Exception as e:
            print(f"Failed embedding for {path}: {e}")
            continue

    return np.array(embeddings)
def detect_image_reuse(article_cluster):
    image_paths = [a["local_image_path"] for a in article_cluster if "local_image_path" in a]

    if len(image_paths) < 2:
        return []

    embeddings = compute_image_embeddings_clip(image_paths)

    reuse_pairs = []
    for i in range(len(embeddings)):
        for j in range(i+1, len(embeddings)):

            v1 = embeddings[i].flatten()
            v2 = embeddings[j].flatten()

            similarity = 1 - cosine(embeddings[i], embeddings[j])
            if similarity > 0.85:
                reuse_pairs.append({
                    "article_1": article_cluster[i]["source"],
                    "article_2": article_cluster[j]["source"],
                    "similarity": similarity,
                    "image_1": image_paths[i],
                    "image_2": image_paths[j]
                })

    return reuse_pairs


In [ ]:
def compute_image_features(article):
    if "local_image_path" not in article:
        article["image_features"] = {"num_images": 0, "alignment_score": 0.5}
        return article

    forensics = analyze_image_forensics_with_vlm(
        article["local_image_path"],
        article["headline"]
    )

    color_emotion = analyze_color_emotion(article["local_image_path"])

    article["image_features"] = {
        "num_images": 1,
        "vlm_analysis": forensics["vlm_analysis"],
        "alignment_score": forensics["alignment_score"], # Pass this to veracity score
        "color_features": color_emotion["color_features"],
        "emotion_dimensions": color_emotion["emotion_dimensions"],
        "sensationalism_from_color": color_emotion["sensationalism_from_color"]
    }
    return article

In [ ]:
from sentence_transformers import util

def compute_claim_similarity_matrix(article_cluster):
    embedder = SentenceTransformer('all-MiniLM-L6-v2')

    all_claims = []
    claim_to_article = {}

    for article in article_cluster:
        for claim in article["text_features"].get("claims", []):
            all_claims.append(claim)
            claim_to_article[len(all_claims) - 1] = article["source"]

    if len(all_claims) < 2:
        return None

    claim_embeddings = embedder.encode(all_claims, convert_to_tensor=True)

    similarity_matrix = util.pytorch_cos_sim(claim_embeddings, claim_embeddings)

    # Identify matching claims (>0.8 similarity) vs contradictions
    matches = []
    contradictions = []

    for i in range(len(all_claims)):
        for j in range(i+1, len(all_claims)):
            sim = similarity_matrix[i][j].item()
            if sim > 0.8:
                matches.append({
                    "claim_1": all_claims[i],
                    "claim_2": all_claims[j],
                    "source_1": claim_to_article[i],
                    "source_2": claim_to_article[j],
                    "similarity": sim,
                    "type": "match"
                })

    return {
        "total_claims": len(all_claims),
        "claim_matches": matches,
        "similarity_matrix": similarity_matrix
    }

def detect_factual_contradictions(article_cluster):
    client = Groq(api_key="api_key")

    all_claims_text = []
    for article in article_cluster:
        claims = article["text_features"].get("claims", [])
        for claim in claims:
            all_claims_text.append(f"{article['source']}: {claim}")

    if len(all_claims_text) < 2:
        return []

    prompt = f"""
    Review these claims from different news sources about the same event.
    Identify any contradictions or disagreements about facts, numbers, dates, etc.

    Claims:
    {chr(10).join(all_claims_text)}

    For each contradiction, specify:
    - The two conflicting claims
    - Which source makes each claim
    - What the contradiction is

    Format:
    CONTRADICTION 1:
      Claim A: [source]: [claim text]
      Claim B: [source]: [claim text]
      Issue: [what contradicts]

    If no contradictions found, respond: "NO CONTRADICTIONS FOUND"
    """

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=500
    )

    return response.choices[0].message.content#parse response


In [ ]:
def check_text_image_alignment(article):
    if "text_features" not in article or "image_features" not in article:
        return None

    text_emotion = article["text_features"]["emotion_analysis"]
    image_emotion = article["image_features"]["emotion_dimensions"] if article["image_features"]["num_images"] > 0 else None

    if not image_emotion:
        return {"alignment_score": 1.0, "mismatch_detected": False}

    text_valence = text_emotion["emotion_distribution"].get("joy", 0) - text_emotion["emotion_distribution"].get("sadness", 0)
    image_valence = image_emotion.get("valence", 0.5) - (1 - image_emotion.get("valence", 0.5))

    valence_mismatch = abs(text_valence - image_valence)

    # If text is negative but image is very positive (or vice versa), flag
    alignment_score = 1.0 - min(valence_mismatch, 1.0)
    mismatch_detected = alignment_score < 0.6

    return {
        "alignment_score": alignment_score,
        "mismatch_detected": mismatch_detected,
        "text_valence": text_valence,
        "image_valence": image_valence,
        "manipulation_risk": "HIGH" if mismatch_detected else "LOW"
    }


In [ ]:
def compute_cluster_consistency_metrics(article_cluster):
    similarity_analysis = compute_claim_similarity_matrix(article_cluster)
    contradictions = detect_factual_contradictions(article_cluster)

    avg_sensationalism = np.mean([
        a["text_features"]["sensationalism_score"]
        for a in article_cluster
    ])

    avg_neutrality = np.mean([
        a["text_features"]["neutrality_score"]
        for a in article_cluster
    ])

    image_reuse_info = detect_image_reuse(article_cluster)
    embedder = SentenceTransformer('all-MiniLM-L6-v2')
    texts = [f"{a['headline']} {a['body_text'][:500]}" for a in article_cluster]
    embeddings = embedder.encode(texts, convert_to_tensor=True)

    avg_text_overlap = 0
    count = 0
    for i in range(len(embeddings)):
        for j in range(i+1, len(embeddings)):
            sim = util.pytorch_cos_sim(embeddings[i], embeddings[j]).item()
            avg_text_overlap += sim
            count += 1
    avg_text_overlap = avg_text_overlap / count if count > 0 else 0

    return {
        "cluster_avg_sensationalism": avg_sensationalism,
        "cluster_avg_neutrality": avg_neutrality,
        "claim_similarity_analysis": similarity_analysis,
        "factual_contradictions": contradictions,
        "image_reuse_detected": len(image_reuse_info) > 0,
        "num_image_reuses": len(image_reuse_info),
        "avg_text_overlap": avg_text_overlap
    }


In [ ]:
from sklearn.preprocessing import MinMaxScaler

def normalize_article_features(article_cluster):
    feature_ranges = {
        "sensationalism_score": [],
        "neutrality_score": [],
        "bias_score": [],
        "num_claims": [],
        "color_sensationalism": [],
        "image_emotion_intensity": [],
        "text_image_mismatch": []
    }

    for article in article_cluster:
        feature_ranges["sensationalism_score"].append(
            article["text_features"]["sensationalism_score"]
        )
        feature_ranges["neutrality_score"].append(
            article["text_features"]["neutrality_score"]
        )
        feature_ranges["bias_score"].append(
            article["text_features"]["bias_score"]
        )
        feature_ranges["num_claims"].append(
            len(article["text_features"]["claims"])
        )

        if article["image_features"]["num_images"] > 0:
            feature_ranges["color_sensationalism"].append(
                article["image_features"]["sensationalism_from_color"]
            )
            feature_ranges["image_emotion_intensity"].append(
                article["image_features"]["emotion_dimensions"]["arousal"]
            )

        mismatch = check_text_image_alignment(article)
        if mismatch:
            feature_ranges["text_image_mismatch"].append(
                1.0 - mismatch["alignment_score"]
            )

    feature_stats = {}
    for feature, values in feature_ranges.items():
        if values:
            feature_stats[feature] = {
                "min": min(values),
                "max": max(values),
                "mean": np.mean(values)
            }

    for article in article_cluster:
        article["normalized_features"] = {}

        article["normalized_features"]["sensationalism"] = article["text_features"]["sensationalism_score"]
        article["normalized_features"]["neutrality"] = article["text_features"]["neutrality_score"]
        article["normalized_features"]["bias"] = article["text_features"]["bias_score"]

        if "num_claims" in feature_stats:
            min_val = feature_stats["num_claims"]["min"]
            max_val = feature_stats["num_claims"]["max"]
            norm_claims = (len(article["text_features"]["claims"]) - min_val) / (max_val - min_val + 1e-6) if max_val > min_val else 0.5
            article["normalized_features"]["normalized_claims"] = norm_claims

        if article["image_features"]["num_images"] > 0:
            article["normalized_features"]["color_sensationalism"] = \
                article["image_features"]["sensationalism_from_color"]
            article["normalized_features"]["image_arousal"] = \
                article["image_features"]["emotion_dimensions"]["arousal"]

        mismatch = check_text_image_alignment(article)
        if mismatch:
            article["normalized_features"]["text_image_alignment"] = mismatch["alignment_score"]


In [ ]:
def compute_veracity_score(article, cluster_consistency, weights=None):
    if weights is None:
        weights = {
            "neutrality": 0.20,
            "low_bias": 0.20,
            "contextual_sensationalism": 0.25, # Adjusted weight
            "text_image_alignment": 0.15,
            "factual_consistency": 0.20
        }

    normalized = article.get("normalized_features", {})

    neutrality_score = normalized.get("neutrality", 0.5)
    bias_score = 1.0 - normalized.get("bias", 0.5)
    article_sensationalism = normalized.get("sensationalism", 0.5)
    cluster_avg = cluster_consistency.get("cluster_avg_sensationalism", 0.5)

    sensationalism_veracity = 1.0 - max(0, article_sensationalism - cluster_avg)

    alignment_score = normalized.get("text_image_alignment", 0.5)

    consistency_score = 0.5
    if cluster_consistency and cluster_consistency.get("claim_similarity_analysis"):
        article_claims = article["text_features"].get("claims", [])
        matches_count = 0
        for match in cluster_consistency["claim_similarity_analysis"].get("claim_matches", []):
            if match["source_1"] == article["source"] or match["source_2"] == article["source"]:
                matches_count += 1
        if article_claims:
            consistency_score = min(matches_count / len(article_claims), 1.0)

    veracity = (
        weights["neutrality"] * neutrality_score +
        weights["low_bias"] * bias_score +
        weights["contextual_sensationalism"] * sensationalism_veracity +
        weights["text_image_alignment"] * alignment_score +
        weights["factual_consistency"] * consistency_score
    )

    return veracity

def rank_articles_by_veracity(article_cluster):
    cluster_consistency = compute_cluster_consistency_metrics(article_cluster)

    trusted_sources = ["pib.gov.in", "reuters.com", "apnews.com", "bbc.com"]
    benchmark = next((a for a in article_cluster if any(s in a['url'] for s in trusted_sources)), None)

    for article in article_cluster:
        if benchmark and article['source'] != benchmark['source']:
            pass

        article["veracity_score"] = compute_veracity_score(article, cluster_consistency)

    ranked = sorted(article_cluster, key=lambda a: a["veracity_score"], reverse=True)
    return ranked, cluster_consistency

In [ ]:
def generate_article_explanation(article, rank, total_articles, cluster_consistency):
    explanation = {
        "article": article["source"],
        "url": article["url"],
        "headline": article["headline"],
        "rank": rank,
        "total_articles": total_articles,
        "veracity_score": round(article["veracity_score"], 3),
        "verdict": "HIGHLY CREDIBLE" if article["veracity_score"] > 0.75
                  else "CREDIBLE" if article["veracity_score"] > 0.6
                  else "QUESTIONABLE" if article["veracity_score"] > 0.45
                  else "UNRELIABLE",
        "factors": {
            "neutrality": {
                "score": round(article["normalized_features"].get("neutrality", 0.5), 2),
                "interpretation": "Article maintains neutral, balanced tone" if article["normalized_features"].get("neutrality", 0.5) > 0.6 else "Article uses emotional/charged language"
            },
            "sensationalism": {
                "score": round(1.0 - article["normalized_features"].get("sensationalism", 0.5), 2),
                "interpretation": "Avoids sensational framing" if article["normalized_features"].get("sensationalism", 0.5) < 0.4 else "Uses sensational headlines/language"
            },
            "bias_markers": {
                "score": round(1.0 - article["normalized_features"].get("bias", 0.5), 2),
                "interpretation": "Low bias detected" if article["normalized_features"].get("bias", 0.5) < 0.4 else "Potential bias or loaded language"
            },
            "image_framing": {
                "score": round(article["normalized_features"].get("text_image_alignment", 0.5), 2),
                "interpretation": "Images aligned with text content" if article["normalized_features"].get("text_image_alignment", 0.5) > 0.6 else "Images may create misleading impression"
            }
        },
        "claims_analysis": {
            "num_claims": len(article["text_features"].get("claims", [])),
            "matching_claims_other_sources": 0,  # Computed below
            "contradicting_claims": []
        }
    }

    if cluster_consistency and cluster_consistency.get("claim_similarity_analysis"):
        for match in cluster_consistency["claim_similarity_analysis"].get("claim_matches", []):
            if match["source_1"] == article["source"] or match["source_2"] == article["source"]:
                explanation["claims_analysis"]["matching_claims_other_sources"] += 1

    if isinstance(cluster_consistency.get("factual_contradictions"), str):
        if article["source"] in cluster_consistency["factual_contradictions"]:
            explanation["claims_analysis"]["contradicting_claims"].append(
                cluster_consistency["factual_contradictions"]
            )

    return explanation


In [ ]:
import json

def generate_cluster_report(ranked_articles, cluster_consistency, topic):
    report = {
        "topic": topic,
        "num_articles_analyzed": len(ranked_articles),
        "timestamp": datetime.now().isoformat(),
        "rankings": []
    }

    for rank, article in enumerate(ranked_articles, 1):
        explanation = generate_article_explanation(
            article, rank, len(ranked_articles), cluster_consistency
        )
        report["rankings"].append(explanation)

    report["cluster_insights"] = {
        "avg_sensationalism_across_sources": round(cluster_consistency["cluster_avg_sensationalism"], 2),
        "avg_neutrality_across_sources": round(cluster_consistency["cluster_avg_neutrality"], 2),
        "image_reuse_detected": cluster_consistency["image_reuse_detected"],
        "high_text_overlap": cluster_consistency["avg_text_overlap"] > 0.7,
        "text_overlap_score": round(cluster_consistency["avg_text_overlap"], 2),
        "key_contradictions": cluster_consistency["factual_contradictions"][:200] if isinstance(cluster_consistency["factual_contradictions"], str) else ""
    }

    most_credible = ranked_articles[0]["source"] if ranked_articles else "N/A"
    report["recommendation"] = f"For this topic, {most_credible} appears to be the most credible source based on neutral tone, low bias markers, and consistency with other outlets."

    return report

def save_report(report, filename="veracity_report.json"):
    """
    Save report to JSON file.
    """
    with open(filename, 'w') as f:
        json.dump(report, f, indent=2)
    print(f"Report saved to {filename}")


In [ ]:
def evaluate_veracity_ranking(ground_truth_rankings, predicted_rankings):

    from sklearn.metrics import ndcg_score, spearmanr

    true_scores = [item["credibility_score"] for item in ground_truth_rankings]
    pred_scores = [item["veracity_score"] for item in predicted_rankings]

    correlation, p_value = spearmanr(true_scores, pred_scores)

    true_labels = [1 if s > 0.6 else 0 for s in true_scores]
    pred_labels = [1 if s > 0.6 else 0 for s in pred_scores]

    from sklearn.metrics import precision_recall_fscore_support
    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels, pred_labels, average='binary'
    )

    return {
        "spearman_correlation": correlation,
        "p_value": p_value,
        "precision": precision,
        "recall": recall,
        "f1_score": f1
    }


In [ ]:
def run_news_credibility_pipeline(
    topic,
    num_days=1,
    min_cluster_size=3,
    save_json=True
):
    print(f"\n Fetching news for topic: {topic}")
    articles = fetch_news_by_topic(topic, num_days=num_days)

    print(f"\n Fetching news for topic: {topic}")
    articles = fetch_news_by_topic(topic, num_days=num_days)

    if not articles:
        print(" No articles found at all.")
        return None

    print(f" Collected {len(articles)} article(s)")


    print(f" Collected {len(articles)} articles")

    print("\n Clustering articles by event...")
    clusters = cluster_articles_by_topic(articles)

    if not clusters:
        print(" No valid clusters found.")
        return None

    print(f" Found {len(clusters)} event cluster(s)")

    all_reports = []

    for cluster_id, cluster in enumerate(clusters, start=1):
        print(f"\n Processing Cluster {cluster_id} ({len(cluster)} articles)")

        download_and_process_images(cluster)

        for i, article in enumerate(cluster, start=1):
            print(f"   Processing article {i}/{len(cluster)} from {article['source']}")
            article["body_text"] = preprocess_text(article.get("body_text", ""))

            compute_text_features(article)
            compute_image_features(article)

        normalize_article_features(cluster)

        ranked_articles, cluster_consistency = rank_articles_by_veracity(cluster)

        report = generate_cluster_report(
            ranked_articles,
            cluster_consistency,
            topic
        )

        all_reports.append(report)

        # Save report
        if save_json:
            filename = f"credibility_report_{topic.replace(' ', '_')}_cluster{cluster_id}.json"
            save_report(report, filename)

        print("\n Top credible source:")
        print(
            ranked_articles[0]["source"],
            "→ Score:",
            round(ranked_articles[0]["veracity_score"], 3)
        )

    print("\n Pipeline execution complete.")
    return all_reports


In [ ]:
from datetime import datetime

In [ ]:
reports = run_news_credibility_pipeline(
    topic="Iran war",
    num_days=1
)


In [ ]:
if reports:
    for cluster_id, report in enumerate(reports, 1):
        print(f"\n" + "="*100)
        print(f" FORENSIC AUDIT: {report['topic']} | CLUSTER {cluster_id}")
        print(f" CLUSTER BASELINE SENSATIONALISM: {report['cluster_insights']['avg_sensationalism_across_sources']}")
        print("="*100)

        graph_data = []

        for item in report['rankings']:
            factors = item['factors']
            v_score = item['veracity_score']

            reasons = []
            if factors['neutrality']['score'] > 0.7: reasons.append("high neutral tone")
            if factors['image_framing']['score'] > 0.7: reasons.append("strong visual alignment")
            if item['claims_analysis']['matching_claims_other_sources'] > 0: reasons.append("factual consensus")
            if factors['bias_markers']['score'] < 0.4: reasons.append("bias markers detected")

            reason_str = "Score driven by " + (", ".join(reasons) if reasons else "moderate across-the-board metrics")

            print(f" SOURCE: {item['article'].upper()}")
            print(f" HEADLINE: {item['headline']}")
            print(f" FORMULATION & VALUES:")
            print(f"   [Veracity: {v_score}] = (Neut: {factors['neutrality']['score']} * 0.20) + "
                  f"(Bias_Resist: {factors['bias_markers']['score']} * 0.20) + "
                  f"(Sens_Resist: {factors['sensationalism']['score']} * 0.25) + "
                  f"(Img_Align: {factors['image_framing']['score']} * 0.15) + "
                  f"(Consensus: {item['claims_analysis']['matching_claims_other_sources']} * 0.20)")
            print(f" VERDICT: {item['verdict']} | {reason_str}")
            print("-" * 100)

            graph_data.append({
                'Source': item['article'],
                'Veracity': v_score,
                'Raw_Sensationalism': 1.0 - factors['sensationalism']['score'], # Extracting back the raw intensity
                'Img_Alignment': factors['image_framing']['score'],
                'Verified_Claims': item['claims_analysis']['num_claims']
            })

        df = pd.DataFrame(graph_data)
        fig, axes = plt.subplots(1, 2, figsize=(16, 5))

        sns.barplot(x='Source', y='Raw_Sensationalism', data=df, ax=axes[0], palette="Reds_r")
        axes[0].axhline(report['cluster_insights']['avg_sensationalism_across_sources'],
                        ls='--', color='blue', label='Cluster Baseline')
        axes[0].set_title(f"Effectiveness: Contextual Emotion Check (Cluster {cluster_id})")
        axes[0].legend()

        sns.lineplot(x='Source', y='Veracity', data=df, marker='o', ax=axes[1], label='Veracity Score', color='green')
        sns.lineplot(x='Source', y='Img_Alignment', data=df, marker='s', ax=axes[1], label='Image Alignment', color='orange')
        axes[1].set_title(f"Multimodal Process: Score vs. Visual Evidence")
        axes[1].set_ylim(0, 1)
        axes[1].legend()

        plt.tight_layout()
        plt.show()

else:
    print("No reports to visualize.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def visualize_pipeline_effectiveness(reports):
    """
    Directly processes the 'reports' list to generate effectiveness metrics.
    """
    if not reports:
        print(" No report data found to visualize.")
        return

    all_data = []
    topic_name = reports[0].get('topic', 'News Analysis')

    for report in reports:
        c_baseline = report['cluster_insights']['avg_sensationalism_across_sources']
        for item in report['rankings']:
            all_data.append({
                'Source': item['article'],
                'Veracity': item['veracity_score'],
                'Neutrality': item['factors']['neutrality']['score'],
                'Sensationalism': 1.0 - item['factors']['sensationalism']['score'], # Raw Intensity
                'Bias': 1.0 - item['factors']['bias_markers']['score'], # Raw Bias
                'Img_Align': item['factors']['image_framing']['score'],
                'Claims': item['claims_analysis']['num_claims'],
                'Cluster_Baseline': c_baseline
            })

    df = pd.DataFrame(all_data)
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    fig.suptitle(f"System Effectiveness Dashboard: {topic_name}", fontsize=22, fontweight='bold')
    sns.set_theme(style="whitegrid")

    # --- PLOT 1: Proof of Setback 2 (Context-Aware Emotion) ---
    # Shows if emotional articles were unfairly penalized or handled correctly
    sns.scatterplot(x='Cluster_Baseline', y='Sensationalism', hue='Source', size='Veracity',
                    data=df, ax=axes[0,0], sizes=(100, 600), alpha=0.7)
    axes[0,0].plot([0, 1], [0, 1], ls="--", c=".3", label="Ideal Contextual Match")
    axes[0,0].set_title("Contextual Emotion Mapping\n(Articles near dashed line show 'Justified Emotion')")
    axes[0,0].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize='small')

    # --- PLOT 2: Process Logic (Feature Correlation) ---
    # Proves the multimodal weighting is working
    corr_matrix = df[['Veracity', 'Neutrality', 'Sensationalism', 'Img_Align', 'Claims']].corr()
    sns.heatmap(corr_matrix, annot=True, cmap='RdYlGn', ax=axes[0,1], fmt=".2f")
    axes[0,1].set_title("Process Logic: Multi-Feature Correlation Matrix")

    # --- PLOT 3: Credibility Breakdown by Source ---
    # Average veracity across all appearances in all clusters
    avg_veracity = df.groupby('Source')['Veracity'].mean().sort_values(ascending=False).reset_index()
    sns.barplot(x='Veracity', y='Source', data=avg_veracity, palette='viridis', ax=axes[1,0])
    axes[1,0].set_title("Mean Veracity Score by News Source")

    # --- PLOT 4: Veracity vs. Visual Alignment ---
    # Shows the impact of the VLM forensic check on the final score
    sns.regplot(x='Img_Align', y='Veracity', data=df, ax=axes[1,1], color='teal', scatter_kws={'s':100})
    axes[1,1].set_title("Impact of VLM Forensic Analysis on Veracity\n(Cross-Modal Grounding Verification)")

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

In [ ]:
visualize_pipeline_effectiveness(reports)